# Serve the official N-ATLaS for the Sovereign Gateway

This notebook runs **`NCAIR1/N-ATLaS`** (Llama-3 8B, fine-tuned by NCAIR/Awarri) on a free Colab GPU with **vLLM**, behind an OpenAI-compatible API, and exposes it through a temporary Cloudflare tunnel so the N-ATLAS Sovereign Gateway can use it.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. Accept the model terms at https://huggingface.co/NCAIR1/N-ATLaS while logged in.
3. Create a Hugging Face read token (https://huggingface.co/settings/tokens) and add it in Colab's **Secrets** panel (key icon on the left) as `HF_TOKEN`, with notebook access on.

Then *Runtime → Run all*. The last cell prints the URL and key to give the gateway.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# vLLM serves the model; cloudflared gives it a public HTTPS URL.
!pip -q install vllm bitsandbytes
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import os, secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# A key so only your gateway can call this endpoint.
API_KEY = "natlas-" + secrets.token_urlsafe(18)
os.environ["VLLM_API_KEY"] = API_KEY
print("Upstream key generated.")

In [ ]:
# A T4 has 16 GB, so the 8B model is loaded in 4-bit (bitsandbytes). On an A100/L4, drop the two quantization flags.
import subprocess
cmd = (
    "nohup vllm serve NCAIR1/N-ATLaS "
    "--served-model-name n-atlas-8b "
    "--dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 "
    "--quantization bitsandbytes --load-format bitsandbytes "
    f"--api-key {API_KEY} --port 8000 > vllm.log 2>&1 &"
)
subprocess.run(cmd, shell=True)
print("vLLM starting. Downloading the model takes a few minutes on first run.")

In [ ]:
import time, requests
for i in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=3).ok:
            print("N-ATLaS is up after", i * 10, "seconds"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
# Sanity check in four Nigerian languages, straight to the model.
import requests
H = {"Authorization": f"Bearer {API_KEY}"}
for q in ["Translate 'Good morning, how can I help you today?' to Yoruba.",
          "Translate 'Good morning, how can I help you today?' to Hausa.",
          "Translate 'Good morning, how can I help you today?' to Igbo.",
          "Explain wetin BVN be for Pidgin, short."]:
    r = requests.post("http://localhost:8000/v1/chat/completions", headers=H, json={
        "model": "n-atlas-8b", "temperature": 0.2, "max_tokens": 120,
        "messages": [{"role": "user", "content": q}]}).json()
    print("Q:", q)
    print("A:", r["choices"][0]["message"]["content"].strip())
    print()

In [ ]:
# Public HTTPS tunnel (temporary URL, valid while this notebook runs).
import re, subprocess
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if m:
        url = m.group(0); break
print()
print("================ GIVE THESE TO THE GATEWAY ================")
print("UPSTREAM_URL     =", f"{url}/v1")
print("UPSTREAM_API_KEY =", API_KEY)
print("UPSTREAM_MODEL   = n-atlas-8b")
print("UPSTREAM_LABEL   = NCAIR1/N-ATLaS on vLLM (Colab GPU)")
print("============================================================")

In [ ]:
# Keep this cell running so the tunnel stays open while you demo or record.
import time
while True:
    time.sleep(60)